# Clean Markdown in, good chunks out: Section Anything and pymupdf4llm with LangChain's MarkdownHeaderTextSplitter

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omniwot/section-anything/blob/main/notebooks/markdown-chunks.ipynb)

**What this shows.** Header-aware chunkers such as LangChain's `MarkdownHeaderTextSplitter` and LlamaIndex's
`MarkdownNodeParser` only work as well as the Markdown they are given: if the PDF converter emits flat or missing
headings, every chunk gets the same (or an empty) heading path (open issues, by number: langchain #41184,
llama_index #23411, semchunk #14). This notebook feeds the **same splitter, same settings** two Markdown
versions of three public papers:

- **Path A:** PDF -> Section Anything Free (`tree.to_markdown(body=True)`, no model, $0) -> splitter.
- **Path B:** PDF -> `pymupdf4llm.to_markdown` -> splitter.

and counts, per path: chunks, chunks with a non-empty heading path, chunks whose first content line starts
mid-sentence, and the deepest heading path. Three example chunks per path are printed.

**What it does not claim.** Three papers are a demonstration, not a benchmark, and the counts are simple
heuristics (a lowercase or continuation first character means "mid-sentence"). pymupdf4llm keeps tables, images
and formatting and needs no LLM (OCR is switched off here so both paths read the same text layer); Section Anything's `body=True` Markdown is
headings plus reflowed paragraph text with running headers and footers removed, so it is lighter in content.
Which is better depends on what your chunks need to carry. Retrieval quality is not measured here (see the
Qasper write-up in `docs/rag-qasper.md` for that).

**Versions used when the saved outputs were made:** section-anything 0.1.9,
pymupdf4llm 2.1, langchain-text-splitters 1.1.3,
llama-index-core 0.14.25, Python 3.11.18, CPU only, macOS.

In [1]:
%pip install -q "section-anything==0.1.9" "pymupdf4llm==2.1" "langchain-text-splitters==1.1.3" "llama-index-core==0.14.25"

Note: you may need to restart the kernel to use updated packages.


## 1. Download the papers (the same three as `heading-levels.ipynb`)

In [2]:
# Three public papers, shortest first. All CC BY 4.0; downloaded from arXiv when you run the notebook.
PAPERS = [
    ("2209.07299v2", "Knowledge Is Flat (Chen et al.), 13 pages, no PDF bookmarks; in READoc"),
    ("2304.08485v2", "Visual Instruction Tuning (Liu et al.), 25 pages, PDF bookmarks; the README's demo paper"),
    ("2305.06327v1", "Thermalization, Chaos and Hydrodynamics (Ganapa), 88 pages, PDF bookmarks, nested; in READoc"),
]

import os, time, urllib.request
from pathlib import Path

PDF_DIR = Path("pdfs"); PDF_DIR.mkdir(exist_ok=True)
for arxiv_id, _ in PAPERS:
    path = PDF_DIR / f"{arxiv_id}.pdf"
    if path.exists():
        continue
    req = urllib.request.Request(f"https://arxiv.org/pdf/{arxiv_id}", headers={"User-Agent": "section-anything-notebook"})
    with urllib.request.urlopen(req, timeout=120) as r:
        data = r.read()
    assert data[:5] == b"%PDF-", arxiv_id
    path.write_bytes(data)
    time.sleep(3)  # be gentle with arXiv
print(sorted(p.name for p in PDF_DIR.glob("*.pdf")))

['2209.07299v2.pdf', '2304.08485v2.pdf', '2305.06327v1.pdf']


## 2. Two Markdown paths, one splitter

In [3]:
import re, time
import section_anything, pymupdf4llm
from langchain_text_splitters import MarkdownHeaderTextSplitter
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core import Document

HEADERS = [("#", "h1"), ("##", "h2"), ("###", "h3"), ("####", "h4"), ("#####", "h5"), ("######", "h6")]
splitter = MarkdownHeaderTextSplitter(headers_to_split_on=HEADERS, strip_headers=True)  # same object for both paths
node_parser = MarkdownNodeParser()

def markdown_a(pdf):
    tree = section_anything.section(pdf, mode="free")
    return tree.to_markdown(body=True)

def markdown_b(pdf):
    return pymupdf4llm.to_markdown(pdf, show_progress=False, use_ocr=False)

CONTINUATION = re.compile(r"^[a-z,;:)\]]|^\.|^and |^or |^of ")   # first character says "I continue a sentence"

def first_content_line(text):
    for line in text.splitlines():
        s = line.strip().strip("*_ ")
        if s and not s.startswith("#") and not s.startswith("|") and not s.startswith("!["):
            return s
    return ""

def path_stats(md):
    chunks = splitter.split_text(md)
    with_path = [c for c in chunks if c.metadata]
    mid = [c for c in chunks if CONTINUATION.match(first_content_line(c.page_content))]
    depth = max((len(c.metadata) for c in chunks), default=0)
    nodes = node_parser.get_nodes_from_documents([Document(text=md)])
    nodes_with_path = [n for n in nodes if n.metadata.get("header_path", "/").strip("/")]
    return chunks, {
        "chunks": len(chunks),
        "with heading path": len(with_path),
        "first line mid-sentence": len(mid),
        "max heading depth": depth,
        "LlamaIndex nodes": len(nodes),
        "nodes with header_path": len(nodes_with_path),
    }

def show_chunks(chunks, n=3, width=280):
    for c in chunks[:n]:
        path = " > ".join(c.metadata.values()) or "(no heading path)"
        body = " ".join(c.page_content.split())[:width]
        print(f"[{path}]\n  {body}{' ...' if len(c.page_content) > width else ''}\n")

## 3. Run both paths on each paper

In [4]:
rows = []
examples = {}
for arxiv_id, about in PAPERS:
    pdf = str(PDF_DIR / f"{arxiv_id}.pdf")
    print("=" * 100); print(f"{arxiv_id}: {about}"); print("=" * 100)
    for path, make in (("A: Section Anything free", markdown_a), ("B: pymupdf4llm", markdown_b)):
        t = time.time(); md = make(pdf); seconds = time.time() - t
        chunks, s = path_stats(md)
        s = {"paper": arxiv_id, "path": path, "markdown chars": len(md), "seconds": round(seconds, 1), **s}
        rows.append(s)
        examples[(arxiv_id, path)] = chunks
        print(f"{path}: {s}")
    print()

2209.07299v2: Knowledge Is Flat (Chen et al.), 13 pages, no PDF bookmarks; in READoc


A: Section Anything free: {'paper': '2209.07299v2', 'path': 'A: Section Anything free', 'markdown chars': 55769, 'seconds': 0.7, 'chunks': 22, 'with heading path': 22, 'first line mid-sentence': 0, 'max heading depth': 3, 'LlamaIndex nodes': 26, 'nodes with header_path': 25}


B: pymupdf4llm: {'paper': '2209.07299v2', 'path': 'B: pymupdf4llm', 'markdown chars': 60175, 'seconds': 2.4, 'chunks': 19, 'with heading path': 19, 'first line mid-sentence': 0, 'max heading depth': 3, 'LlamaIndex nodes': 19, 'nodes with header_path': 18}

2304.08485v2: Visual Instruction Tuning (Liu et al.), 25 pages, PDF bookmarks; the README's demo paper


A: Section Anything free: {'paper': '2304.08485v2', 'path': 'A: Section Anything free', 'markdown chars': 78088, 'seconds': 2.9, 'chunks': 18, 'with heading path': 18, 'first line mid-sentence': 0, 'max heading depth': 3, 'LlamaIndex nodes': 19, 'nodes with header_path': 18}


B: pymupdf4llm: {'paper': '2304.08485v2', 'path': 'B: pymupdf4llm', 'markdown chars': 91546, 'seconds': 3.2, 'chunks': 26, 'with heading path': 26, 'first line mid-sentence': 0, 'max heading depth': 4, 'LlamaIndex nodes': 27, 'nodes with header_path': 26}

2305.06327v1: Thermalization, Chaos and Hydrodynamics (Ganapa), 88 pages, PDF bookmarks, nested; in READoc


A: Section Anything free: {'paper': '2305.06327v1', 'path': 'A: Section Anything free', 'markdown chars': 170953, 'seconds': 2.8, 'chunks': 55, 'with heading path': 54, 'first line mid-sentence': 0, 'max heading depth': 2, 'LlamaIndex nodes': 61, 'nodes with header_path': 18}


B: pymupdf4llm: {'paper': '2305.06327v1', 'path': 'B: pymupdf4llm', 'markdown chars': 194676, 'seconds': 8.3, 'chunks': 63, 'with heading path': 63, 'first line mid-sentence': 1, 'max heading depth': 4, 'LlamaIndex nodes': 75, 'nodes with header_path': 55}



## 4. Counts per path

In [5]:
cols = ["paper", "path", "chunks", "with heading path", "first line mid-sentence", "max heading depth", "LlamaIndex nodes", "nodes with header_path", "seconds"]
print("  ".join(f"{c:>{max(len(c), 6)}}" for c in cols))
for r in rows:
    print("  ".join(f"{str(r[c]):>{max(len(c), 6)}}" for c in cols))

 paper    path  chunks  with heading path  first line mid-sentence  max heading depth  LlamaIndex nodes  nodes with header_path  seconds
2209.07299v2  A: Section Anything free      22                 22                        0                  3                26                      25      0.7
2209.07299v2  B: pymupdf4llm      19                 19                        0                  3                19                      18      2.4
2304.08485v2  A: Section Anything free      18                 18                        0                  3                19                      18      2.9
2304.08485v2  B: pymupdf4llm      26                 26                        0                  4                27                      26      3.2
2305.06327v1  A: Section Anything free      55                 54                        0                  2                61                      18      2.8
2305.06327v1  B: pymupdf4llm      63                 63                       

## 5. Three example chunks per path (first paper)

The heading path from the splitter's metadata is in brackets; the chunk text is flattened and cut at 280 characters.

In [6]:
for arxiv_id, _ in PAPERS[:1]:
    for path in ("A: Section Anything free", "B: pymupdf4llm"):
        print(f"##### {arxiv_id}, path {path}: chunks 2-4 of {len(examples[(arxiv_id, path)])}\n")
        show_chunks(examples[(arxiv_id, path)][1:], n=3)

##### 2209.07299v2, path A: Section Anything free: chunks 2-4 of 22

[Knowledge Is Flat: A Seq2Seq Generative Framework for Various Knowledge Graph Completion > Abstract]
  Knowledge Graph Completion (KGC) has been recently extended to multiple knowledge graph (KG) structures, initiating new research static KGC, temporal KGC directions, e.g. and few-shot KGC (Ji et al., 2022). Previous works often design KGC models closely coupled with speciﬁc graph ...

[Knowledge Is Flat: A Seq2Seq Generative Framework for Various Knowledge Graph Completion > 1 Introduction]
  Knowledge graph completion (KGC) has been a fundamental task to discover unobserved facts from various knowledge graph (KG) structures, including static KGC (SKGC), temporal KGC (TKGC) and few-shot KGC (FKGC) (Ji et al., 2022). As shown in Figure 1, TKGC (in orange) contains temp ...

[Knowledge Is Flat: A Seq2Seq Generative Framework for Various Knowledge Graph Completion > 1 Introduction > FB15K-237 and FB15K-237N for SKGC,]


## Reading the counts

- The splitter is the same object with the same six header levels for both paths, and the LlamaIndex parser
  likewise; nothing is tuned per path.
- **Chunks with a heading path** depends only on whether the converter printed Markdown headings above the
  text. On these three papers both paths give every LangChain chunk a path (one exception: the thesis's front
  matter in path A). The LlamaIndex `header_path` counts differ on the thesis (18 of 61 against 55 of 75) because
  `MarkdownNodeParser` records only the *parents* of a node: path A's tree there starts at `###` with chapters and
  their sections at the same level, so most nodes have no parent, while path B's tree has four levels.
- **Max heading depth** shows whether the headings carry levels: 3 and 3 on the first paper, 3 and 4 on the second,
  2 and 4 on the thesis, where Free finds no document title and flattens chapters (see `heading-levels.ipynb`).
- **First line mid-sentence** found almost nothing (0 and 1 chunks): the heuristic looks only at the first
  character. It misses a real case visible in the examples above: path A's chunk
  "1 Introduction > FB15K-237 and FB15K-237N for SKGC," is a line of body text taken as a heading on this
  bookmark-less paper, and the next chunk starts "ICEWS14 for TKGC ..." in the middle of a sentence, with a capital.
  Read the example chunks, not only the counts.
- The chunk counts differ by exactly the heading differences shown in `heading-levels.ipynb`: path A has the
  extra false headings on the bookmark-less paper, path B has the figure and prompt labels on the second paper.
- Path B keeps the bold markers of a heading in the metadata (`**1 Introduction**`); strip them if you build
  filters on the heading path.
- Path A's Markdown is smaller because it contains headings and reflowed paragraph text only (tables, figure
  text and equations stay in the section text; running headers and footers are removed). Path B keeps the
  whole page content, including tables and bold or italic runs, with no LLM; wall-clock seconds are in the table.
  If you need the full page content with Section Anything's headings, `tree.sections` gives each section's
  page span and character span, so any text extractor's output can be sliced by section instead.